# Cross-Framework Comparison - Logistic Regression

In [ ]:
# Set path of the custom jar that contains the new models
import os
os.environ["CAPYMOA_MOA_JAR"] = r"jar_path"

Global variables

In [2]:
DEFAULT_LR = 0.01
DEFAULT_BIAS_LR = 0.01
DEFAULT_L1 = 0.0
DEFAULT_L2 = 0.0
DEFAULT_CLIP = 1e12
DEFAULT_BIAS_INIT = 0.0
MAX_INSTANCES = 100000
SEED = 42

Global imports

In [3]:
from capymoa.evaluation import prequential_evaluation
from capymoa.classifier import LogisticRegression
from river import linear_model, metrics, optim, evaluate
from tabulate import tabulate

Global functions

In [4]:
def adaptStreamForRiver(stream):
    data = []

    for i, instance in enumerate(stream):
        if(i > MAX_INSTANCES): break

        # features
        x = {f"f{j}": float(v) for j, v in enumerate(instance.x)}

        # label
        y = instance.y_index
        
        data.append((x, y))
    
    return data

In [5]:
def evaluateStream(stream_factory, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT):
    # stream_factory must be deterministic. It is the constructor of the stream.

    capyMoaResults = _evaluateStreamOnCapyMoa(stream_factory(), lr, b_lr, l1, l2, clip, bias_init)

    riverStream = adaptStreamForRiver(stream_factory())

    riverResults = _evaluateStreamOnRiver(riverStream, lr, b_lr, l1, l2, clip, bias_init)

    table = []
    for key in ["Accuracy", "F1", "Precision", "Recall"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.2f}%",
            f"{river:.2f}%",
            f"{(capy - river):+.2f}%"
        ])

    print("\n--- Comparison CapyMOA vs River ---\n")
    print(
        tabulate(
            table,
            headers=["Metric", "CapyMOA", "River", "Delta"],
            tablefmt="fancy_grid"
        )
    )

def _evaluateStreamOnCapyMoa(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_capymoa = LogisticRegression(
        schema=stream.get_schema(),
        learning_rate=lr,
        bias_learning_rate=b_lr,
        l1_penalty=l1,
        l2_penalty=l2,
        clip_gradient=clip,
        bias_init=bias_init
    )

    results = prequential_evaluation(
        stream=stream,
        learner=log_reg_capymoa,
        max_instances=MAX_INSTANCES,
    )

    cumulative = results["cumulative"]

    return {
        "Accuracy": cumulative.accuracy(),
        "F1": cumulative.f1_score(),
        "Precision": cumulative.precision(),
        "Recall": cumulative.recall(),
    }

def _evaluateStreamOnRiver(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_river = linear_model.LogisticRegression(
        optimizer=optim.SGD(lr),
        intercept_lr=b_lr,
        l1=l1,
        l2=l2,
        clip_gradient=clip,
        intercept_init=bias_init
    )

    metric = (
        metrics.Accuracy()
        + metrics.F1()
        + metrics.Precision()
        + metrics.Recall()
    )

    score = evaluate.progressive_val_score(
        dataset=stream,
        model=log_reg_river,
        metric=metric
    )

    return {
        "Accuracy": float(score[0].get()*100),
        "F1": float(score[1].get()*100),
        "Precision": float(score[2].get()*100),
        "Recall": float(score[3].get()*100),
    }

## Electricity dataset

In [6]:
from capymoa.datasets import Electricity

evaluateStream(Electricity)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 66.50%    │ 66.50%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 65.01%    │ 55.74%  │ +9.27%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 65.74%    │ 63.48%  │ +2.26%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 64.30%    │ 49.69%  │ +14.61% │
╘═══════════╧═══════════╧═════════╧═════════╛


## ElectricityTiny dataset

In [7]:
from capymoa.datasets import ElectricityTiny

evaluateStream(ElectricityTiny)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 62.85%    │ 62.85%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 58.67%    │ 38.85%  │ +19.82% │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 60.27%    │ 55.79%  │ +4.48%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 57.16%    │ 29.80%  │ +27.36% │
╘═══════════╧═══════════╧═════════╧═════════╛


## RandomRBFGenerator

In [8]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=2,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 85.28%    │ 85.28%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 84.75%    │ 81.59%  │ +3.16%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 85.19%    │ 84.83%  │ +0.36%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 84.31%    │ 78.59%  │ +5.72%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## Hyper100k dataset

In [9]:
from capymoa.datasets import Hyper100k

evaluateStream(Hyper100k)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 89.86%    │ 89.86%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 89.92%    │ 90.15%  │ -0.23%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 89.98%    │ 87.85%  │ +2.13%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 89.85%    │ 92.57%  │ -2.71%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## SEA dataset generator

In [10]:
from capymoa.stream.generator import SEA

def make_stream():
    return SEA(
        instance_random_seed=SEED,
        function=1,
        balance_classes=False,
        noise_percentage=10,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 82.91%    │ 82.91%  │ -0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 81.09%    │ 87.05%  │ -5.96%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 81.92%    │ 84.69%  │ -2.77%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 80.28%    │ 89.56%  │ -9.28%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## HyperPlaneClassification dataset

In [11]:
from capymoa.stream.generator import HyperPlaneClassification

def make_stream():
    return HyperPlaneClassification(
        instance_random_seed=SEED,
        number_of_classes=2,
        number_of_attributes=10,
        number_of_drifting_attributes=2,
        magnitude_of_change=0.0,
        noise_percentage=5,
        sigma_percentage=10,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 89.87%    │ 89.87%  │ -0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 89.94%    │ 90.18%  │ -0.24%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 90.01%    │ 87.71%  │ +2.30%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 89.86%    │ 92.78%  │ -2.92%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## RandomTreeGenerator

In [12]:
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return RandomTreeGenerator(
        instance_random_seed=SEED,
        tree_random_seed=SEED,
        num_classes=2,
        num_nominals=0,
        num_numerics=5,
        max_tree_depth=5,
        first_leaf_level=3,
        leaf_fraction=0.15,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 81.05%    │ 81.06%  │ -0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 77.21%    │ 66.09%  │ +11.12% │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 79.66%    │ 76.99%  │ +2.67%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 74.89%    │ 57.89%  │ +17.01% │
╘═══════════╧═══════════╧═════════╧═════════╛


## Electricity dataset (changed model parameters)

### L2

In [13]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=0.01, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 65.65%    │ 65.65%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 63.93%    │ 51.37%  │ +12.56% │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 65.27%    │ 64.39%  │ +0.88%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 62.65%    │ 42.74%  │ +19.91% │
╘═══════════╧═══════════╧═════════╧═════════╛


### L1

In [14]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=0.01, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 65.02%    │ 65.02%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 63.20%    │ 48.65%  │ +14.55% │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 64.87%    │ 64.57%  │ +0.30%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 61.61%    │ 39.02%  │ +22.59% │
╘═══════════╧═══════════╧═════════╧═════════╛


### Learning rate

In [15]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=0.1, b_lr=0.1, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 76.56%    │ 76.56%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 75.83%    │ 71.25%  │ +4.57%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 76.16%    │ 74.32%  │ +1.84%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 75.49%    │ 68.43%  │ +7.06%  │
╘═══════════╧═══════════╧═════════╧═════════╛


### Gradient clipping

In [16]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=1, bias_init=DEFAULT_BIAS_INIT)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 66.50%    │ 66.50%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 65.01%    │ 55.74%  │ +9.27%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 65.74%    │ 63.48%  │ +2.26%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 64.30%    │ 49.69%  │ +14.61% │
╘═══════════╧═══════════╧═════════╧═════════╛


### Bias initialization

In [17]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=3)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 65.28%    │ 65.28%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 63.73%    │ 54.43%  │ +9.30%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 64.35%    │ 61.47%  │ +2.88%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 63.13%    │ 48.84%  │ +14.29% │
╘═══════════╧═══════════╧═════════╧═════════╛
